In [1]:
import tkinter as tk
import tkinter.filedialog as fd
import numpy as np
import easygui
import openpyxl

In [2]:
def interp(x,x0,x1,y0,y1):
    y = y0 + (x-x0)*((y1-y0)/(x1-x0))
    return y

In [3]:
#Ask for the relevant ID numbers
aID = str(easygui.enterbox('Enter vessel ID number A'))
bID = str(easygui.enterbox('Enter vessel ID number B'))

#Import the velocity data from excel sheets 
tk.Tk().withdraw()
afile = fd.askopenfilename()
bfile = fd.askopenfilename()
bwb = openpyxl.load_workbook(bfile)

In [4]:
#Import the velocity data from excel sheets 
awb = openpyxl.load_workbook(afile)
aws = awb.active
aV = aws['B']
oaV = aV[1:]
aV=[]
for cell in oaV:
    aV.append(cell.value)
aV = np.array(aV)

bws = bwb.active
bV = bws['B']
obV = bV[1:]
bV=[]
for cell in obV:
    bV.append(cell.value)
bV = np.array(bV)

#Import the time data
aTime = aws['A']
oaTime = aTime[1:]
aTime=[]
for cell in oaTime:
    aTime.append(cell.value)
aTime = np.array(aTime)    

bTime = bws['A']
obTime = bTime[1:]
bTime=[]
for cell in obTime:
    bTime.append(cell.value)
bTime = np.array(bTime)

#Ask for radius values
aR = float(easygui.enterbox('Enter relevant radius for A'))
bR = float(easygui.enterbox('Enter relevant radius for B'))

#Convert to Q arrays
aQ = aV * np.pi * aR**2
bQ = bV * np.pi * bR**2 #Ignore units because it all shakes into ratios in the end


In [5]:
#Interrpolate the Q values for the desired time intervals

#Calculate the actual t array
ntime = np.linspace(0, 0.955, num=478)

#Hardcode t == 0
naQ = [aQ[0]]
nbQ = [bQ[0]]
a = 0
b = 0

for time in ntime[1:-2]:
    iA = np.searchsorted(aTime,time)
    aq = interp(time,aTime[iA-1],aTime[iA],aQ[iA-1],aQ[iA])
    naQ = np.append(naQ,aq)
    iB = np.searchsorted(bTime,time)
    bq = interp(time,bTime[iB-1],bTime[iB],bQ[iB-1],bQ[iB])
    nbQ = np.append(nbQ,bq)

naQ = np.append(naQ,aQ[-1])
nbQ = np.append(nbQ,bQ[-1])


In [6]:
aQ=naQ
bQ=nbQ
#Determine the splitting ratios
try:
    aSR = aQ/(aQ+bQ)
    bSR = bQ/(aQ+bQ)
except:
    if aQ.size > bQ.size:
        bQ = np.append(bQ,bQ[-1])
    else:
        aQ= np.append(aQ,aQ[-1])
    aSR = aQ/(aQ+bQ)
    bSR = bQ/(aQ+bQ)
name = aID+';'+bID
SR = [name]

i = 0
for valueA in aSR:
    valueB = bSR[i]
    
    i += 1
    item = str(np.round(valueA,2)) + ';' + str(np.round(valueB,2))
    SR.append(item)   

In [7]:
#Open and write to the excel sheet
file = "C:\\Users\\cbnor\\Documents\\Full Body Flow Model Project\\SplittingRatios.xlsx"
#Rewrite to actual file location as appropriate! 
wb = openpyxl.load_workbook(file,data_only=True)
ws = wb.worksheets[0]
col = ws.max_column+1

i=1
for value in SR:
     ws.cell(i,col).value = value
     i += 1 

wb.save(file)
wb.close()